In [1]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [2]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    334 non-null    object
 1   label   334 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 5.3+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'business' if x == 0 else 'entertainment' if x == 1 else 'politics' if x == 2 else 'sport' if x == 3 else 'tech')

labels = test['label'].unique()

test

,text,label
0,Dogged Federer claims Dubai crown World number...,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business
...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport
330,Budget Aston takes on Porsche British car make...,business
331,Hi-tech posters guide commuters Interactive po...,tech
332,Hotspot users gain free net calls People using...,tech


In [4]:
load_dotenv()

api_key=os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [5]:
def classify(text, labels):

    sys_instruct="You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."

    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_CIVIC_INTEGRITY",
                threshold="BLOCK_NONE"
            ),
            ],
        ),
        contents=f"Classify the following text based on the task: Category classification of news articles. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"
    )

    request_time = time.time() - start_time
    completion = response.text
    if completion:
        completion = completion.lower()
    else:
        completion = "None"
    completion_tokens = response.usage_metadata.candidates_token_count
    prompt_tokens = response.usage_metadata.prompt_token_count
    total_tokens = response.usage_metadata.total_token_count

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'business' in text:
        return 'business'
    if 'entertainment' in text:
        return 'entertainment'
    if 'politics' in text:
        return 'politics'
    if 'sport' in text:
        return 'sport'
    if 'tech' in text:
        return 'tech'
    else:
        return 'error'

In [6]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_gemini_ZS_multiclass2.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/gemini_ZS_multiclass2.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,Dogged Federer claims Dubai crown World number...,sport,sport\n,1.405618,2.0,518.0,520.0,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics,politics\n,1.620158,2.0,345.0,347.0,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment,entertainment\n,1.185920,2.0,367.0,369.0,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport,sport\n,1.238078,2.0,624.0,626.0,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business,business\n,1.347132,2.0,699.0,701.0,business
...,...,...,...,...,...,...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport,sport\n,0.496108,2.0,885.0,887.0,sport
330,Budget Aston takes on Porsche British car make...,business,business\n,0.516608,2.0,373.0,375.0,business
331,Hi-tech posters guide commuters Interactive po...,tech,tech\n,1.243595,2.0,481.0,483.0,tech
332,Hotspot users gain free net calls People using...,tech,tech\n,1.122938,2.0,403.0,405.0,tech


In [7]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,Dogged Federer claims Dubai crown World number...,sport,sport\n,1.405618,2.0,518.0,520.0,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics,politics\n,1.620158,2.0,345.0,347.0,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment,entertainment\n,1.185920,2.0,367.0,369.0,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport,sport\n,1.238078,2.0,624.0,626.0,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business,business\n,1.347132,2.0,699.0,701.0,business
...,...,...,...,...,...,...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport,sport\n,0.496108,2.0,885.0,887.0,sport
330,Budget Aston takes on Porsche British car make...,business,business\n,0.516608,2.0,373.0,375.0,business
331,Hi-tech posters guide commuters Interactive po...,tech,tech\n,1.243595,2.0,481.0,483.0,tech
332,Hotspot users gain free net calls People using...,tech,tech\n,1.122938,2.0,403.0,405.0,tech


In [8]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.979042
F1 score: 0.978933
Precision: 0.979959
Recall: 0.979042


In [9]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 1.0827174871981502
Average completion tokens: 1.9970059880239521
Average prompt tokens: 566.4281437125749
Average total tokens: 568.4251497005988


In [10]:
input_token_price = 0.1/1_000_000
output_token_price = 0.4/1_000_000

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.019185500000000005


In [11]:
with open('results/gemini_ZS_multiclass2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')